# Zero-Shot Irony Classification — OLMo2 Results

**Approach**: OLMo2 is a decoder-only (causal LM) model that returns free-text answers.  
Zero-shot prediction is extracted by parsing the model's natural-language output:

1. Each `model_output` is searched for a clear **yes / no** signal.
2. The first line is matched with a regex; broader text search is the fallback.
3. A **proxy score** is computed from keyword counts:  
   `score = count("yes"/"is ironic") − count("no"/"is not ironic")`  
   (positive → more ironic signals, negative → more non-ironic signals)
4. Items whose output was **truncated** before a verdict (mainly C3 chain-of-thought  
   prompts) are flagged as `uncertain` and excluded from metrics.

Output is formatted identically to `results_zeroshot.txt` produced by the ModernBERT notebook.

## Step 1 — Imports & Configuration

In [1]:
import re
import json
import numpy as np
import pandas as pd
from datetime import datetime
from sklearn.metrics import f1_score, accuracy_score

# ── Configuration ─────────────────────────────────────────────────────────
CFG = {
    "c1_path" : "results_gemma_zeroshot_condition1.csv",
    "c2_path" : "results_gemma_zeroshot_condition2.csv",
    "c3_path" : "results_gemma_zeroshot_condition3.csv",
    "output"  : "results_gemma_zeroshot.txt",
}

RESULTS_FILE = CFG["output"]

def log(text=''):
    print(text)
    with open(RESULTS_FILE, "a", encoding="utf-8") as f:
        f.write(text + "\n")

# Clear / initialise results file
with open(RESULTS_FILE, "w", encoding="utf-8") as f:
    f.write("Zero-Shot Irony Classification Results (OLMo2)\n")
    f.write(f"Run started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write("=" * 60 + "\n\n")

print(f"Results will be written to: {RESULTS_FILE}")

Results will be written to: results_gemma_zeroshot.txt


## Step 2 — Load Data

In [2]:
c1 = pd.read_csv(CFG["c1_path"])
c2 = pd.read_csv(CFG["c2_path"])
c3 = pd.read_csv(CFG["c3_path"])

# Normalise column names to a common schema
c1 = c1.rename(columns={"context-level": "level", "basic_utterance": "base_utterance"})
c2 = c2.rename(columns={"cg_level": "level"})
c3 = c3.rename(columns={"prompt_level": "level"})

c1["condition"] = "C1_context_richness"
c2["condition"] = "C2_common_ground"
c3["condition"] = "C3_prompting_style"

COLS = ["item_id", "condition", "level", "irony_label", "base_utterance",
        "full_prompt", "model_output"]
df = pd.concat([c1[COLS], c2[COLS], c3[COLS]], ignore_index=True)
df = df.dropna(subset=["irony_label", "model_output"]).reset_index(drop=True)
df["label"] = (df["irony_label"] == "ironic").astype(int)  # ironic=1, non-ironic=0

print(f"Total rows: {len(df)}")
print(df[["condition", "level", "irony_label"]].value_counts().sort_index().to_string())

Total rows: 339
condition            level  irony_label
C1_context_richness  high   ironic         30
                            non-ironic     30
                     low    ironic         30
                            non-ironic     30
C2_common_ground     high   ironic         30
                            non-ironic     29
                     low    ironic         30
                            non-ironic     30
C3_prompting_style   high   ironic         25
                            non-ironic     25
                     low    ironic         25
                            non-ironic     25


## Step 3 — Prediction Parser

OLMo2 answers in free text, so we extract the verdict with a three-stage heuristic:

| Stage | What it checks |
|-------|---------------|
| 1 | First line starts with **"Yes"** or **"No"** |
| 2 | First line contains **"is ironic"** / **"is not ironic"** |
| 3 | Keyword balance over the full output |

Items with no detectable signal are marked **uncertain** (`pred = -1`).  
This happens for C3 chain-of-thought outputs that were truncated before a verdict.

In [3]:
def _keyword_score(text: str) -> float:
    """proxy score = #(ironic signals) - #(non-ironic signals)"""
    lower = text.lower()
    yes_hits = len(re.findall(r'\byes\b|\bis ironic\b', lower))
    no_hits  = len(re.findall(r'\bno\b|\bis not ironic\b|\bnon.ironic\b', lower))
    return float(yes_hits - no_hits)


def parse_prediction(text: str):
    """
    Returns
    -------
    pred  : int    — 1 (ironic), 0 (non-ironic), -1 (uncertain / truncated)
    score : float  — proxy confidence (positive → ironic, negative → non-ironic)
    """
    text  = str(text).strip()
    first = text.split("\n")[0].lower()

    # Stage 1: explicit yes/no opening
    if re.match(r'^\s*yes[\s,.]', first):
        return 1, max(_keyword_score(text), 0.1)
    if re.match(r'^\s*no[\s,.]', first):
        return 0, min(_keyword_score(text), -0.1)

    # Stage 2: irony phrase in first line
    if re.search(r'\bis ironic\b', first) and not re.search(r'not ironic', first):
        return 1, _keyword_score(text)
    if re.search(r'is not ironic|non.ironic', first):
        return 0, _keyword_score(text)

    # Stage 3: keyword balance over full text
    score = _keyword_score(text)
    if score > 0:
        return 1, score
    if score < 0:
        return 0, score

    return -1, 0.0  # uncertain / truncated


results     = df["model_output"].apply(parse_prediction)
df["pred"]  = [r[0] for r in results]
df["score"] = [r[1] for r in results]

n_uncertain = (df["pred"] == -1).sum()
print(f"Uncertain / truncated outputs : {n_uncertain} / {len(df)}")
if n_uncertain:
    print("  (Excluded from metrics; marked 'uncertain' in output file)")

print("\nPrediction distribution per condition:")
print(df.groupby(["condition", "pred"]).size().to_string())

Uncertain / truncated outputs : 20 / 339
  (Excluded from metrics; marked 'uncertain' in output file)

Prediction distribution per condition:
condition            pred
C1_context_richness   1      120
C2_common_ground      0        2
                      1      117
C3_prompting_style   -1       20
                      1       80


## Step 4 — Condition-Level Metrics

In [4]:
label_map = {1: "ironic", 0: "non-ironic", -1: "uncertain"}
condition_results = {}

log("[ZERO-SHOT RESULTS — CONDITION SUMMARY]")
log(f"{'Condition':<25}  {'N':>4}  {'Macro F1':>9}  {'Accuracy':>9}  "
    f"{'F1 ironic':>10}  {'F1 non-ironic':>14}")
log("-" * 82)

for cond_name, group in df.groupby("condition"):
    grp = group[group["pred"] != -1].copy()
    condition_results[cond_name] = group  # keep all rows for per-item output

    if len(grp) == 0:
        log(f"{cond_name:<25}  {len(group):>4}  {'N/A':>9}  {'N/A':>9}  {'N/A':>10}  {'N/A':>14}")
        continue

    labels       = grp["label"].tolist()
    preds        = grp["pred"].tolist()
    macro_f1     = f1_score(labels, preds, average="macro")
    acc          = accuracy_score(labels, preds)
    f1_ironic    = f1_score(labels, preds, pos_label=1, average="binary")
    f1_nonironic = f1_score(labels, preds, pos_label=0, average="binary")

    log(f"{cond_name:<25}  {len(group):>4}  {macro_f1:>9.4f}  {acc:>9.4f}  "
        f"{f1_ironic:>10.4f}  {f1_nonironic:>14.4f}")

log()

[ZERO-SHOT RESULTS — CONDITION SUMMARY]
Condition                     N   Macro F1   Accuracy   F1 ironic   F1 non-ironic
----------------------------------------------------------------------------------
C1_context_richness         120     0.3333     0.5000      0.6667          0.0000
C2_common_ground            119     0.3497     0.5042      0.6667          0.0328
C3_prompting_style          100     0.3333     0.5000      0.6667          0.0000



## Step 5 — Per-Level Breakdown

In [5]:
log("[ZERO-SHOT RESULTS — PER LEVEL BREAKDOWN]")
log(f"{'Condition':<25}  {'Level':<8}  {'N':>4}  {'Macro F1':>9}  {'Accuracy':>9}")
log("-" * 65)

for cond_name, group in condition_results.items():
    for level, grp in group.groupby("level"):
        valid = grp[grp["pred"] != -1]
        if len(valid) == 0:
            log(f"{cond_name:<25}  {level:<8}  {len(grp):>4}  {'N/A':>9}  {'N/A':>9}")
            continue
        lf1  = f1_score(valid["label"], valid["pred"], average="macro")
        lacc = accuracy_score(valid["label"], valid["pred"])
        log(f"{cond_name:<25}  {level:<8}  {len(grp):>4}  {lf1:>9.4f}  {lacc:>9.4f}")

log()

[ZERO-SHOT RESULTS — PER LEVEL BREAKDOWN]
Condition                  Level        N   Macro F1   Accuracy
-----------------------------------------------------------------
C1_context_richness        high        60     0.3333     0.5000
C1_context_richness        low         60     0.3333     0.5000
C2_common_ground           high        59     0.3656     0.5085
C2_common_ground           low         60     0.3333     0.5000
C3_prompting_style         high        50     0.3333     0.5000
C3_prompting_style         low         50     0.3333     0.5000



## Step 6 — Per-Item Predictions

In [6]:
log("[ZERO-SHOT RESULTS — PER ITEM PREDICTIONS]")

for cond_name, group in condition_results.items():
    log(f"\n  Condition: {cond_name}")
    log(f"  {'item_id':<15}  {'level':<8}  {'true label':<12}  "
        f"{'predicted':<12}  {'score':>8}  {'correct'}")
    log("  " + "-" * 72)

    for _, row in group.iterrows():
        pred_label  = label_map[row["pred"]]
        correct_str = "N/A" if row["pred"] == -1 else ("YES" if row["label"] == row["pred"] else "NO")
        log(f"  {row['item_id']:<15}  {row['level']:<8}  "
            f"{label_map[row['label']]:<12}  "
            f"{pred_label:<12}  "
            f"{row['score']:>+8.3f}  {correct_str}")

log()
log(f"Run finished: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"\nAll results saved to: {RESULTS_FILE}")

[ZERO-SHOT RESULTS — PER ITEM PREDICTIONS]

  Condition: C1_context_richness
  item_id          level     true label    predicted        score  correct
  ------------------------------------------------------------------------
  C1_01_L          low       ironic        ironic          +1.000  YES
  C1_01_L_NI       low       non-ironic    ironic          +1.000  NO
  C1_01_H          high      ironic        ironic          +1.000  YES
  C1_01_H_NI       high      non-ironic    ironic          +1.000  NO
  C1_02_L          low       ironic        ironic          +2.000  YES
  C1_02_L_NI       low       non-ironic    ironic          +2.000  NO
  C1_02_H          high      ironic        ironic          +1.000  YES
  C1_02_H_NI       high      non-ironic    ironic          +1.000  NO
  C1_03_L          low       ironic        ironic          +1.000  YES
  C1_03_L_NI       low       non-ironic    ironic          +1.000  NO
  C1_03_H          high      ironic        ironic          +1.000  Y

## Notes

### Score column
`score = count("yes" / "is ironic") − count("no" / "is not ironic")`  
- Positive → output leans ironic  
- Negative → output leans non-ironic  
- `0.000` → uncertain / truncated (no verdict extracted)

Unlike ModernBERT logit differences, this is a **keyword-count proxy** — magnitudes  
are not directly comparable across conditions.

### Uncertain / truncated items (C3)
C3 used chain-of-thought prompts. OLMo2's outputs were truncated before the final  
yes/no verdict (`max_new_tokens` was likely too small for the reasoning chain).  
These items are:
- Included in the per-item table with `predicted = uncertain`
- **Excluded** from F1 / accuracy calculations
- Marked `correct = N/A`